## Using nilearn to visualize NBS output

In [1]:
import scipy.io as sio
import numpy as np
    
nbs = sio.loadmat("C:\\Users\\tempu\\Downloads\\research\\labs\\gratton\\Arousal-Project\\nbs\\nbs_tests\\t-test_thres4_Apos_btw.mat", simplify_cells=True);

Extract variables

In [2]:
# Extract the fields
node_coords = nbs['nbs']['NBS']['node_coor']   # shape: (n_nodes, 3)
con_mat     = nbs['nbs']['NBS']['con_mat'].toarray()      # shape: (n_nodes, n_nodes)
node_labels = nbs['nbs']['NBS']['node_label']   # list of region names

Checks

In [3]:
raw  = sio.loadmat("C:\\Users\\tempu\\Downloads\\research\\labs\\gratton\\Arousal-Project\\nbs\\nbs_tests\\t-test_thres4_Apos_btw.mat", simplify_cells=True);

# Inspect the top level
print(type(raw['nbs']))
print(raw['nbs'].keys())

# Dig into NBS
nbs_struct = raw['nbs']['NBS']
print(type(nbs_struct))
print(nbs_struct.keys())  # shows all field names

# Look at con_mat specifically
con_mat_raw = nbs_struct['con_mat']
print(type(con_mat_raw))
print(con_mat_raw.shape)
print(con_mat_raw.dtype)

<class 'dict'>
dict_keys(['GLM', 'STATS', 'NBS', 'UI'])
<class 'dict'>
dict_keys(['node_coor', 'node_label', 'n', 'con_mat', 'pval', 'test_stat'])
<class 'scipy.sparse._csc.csc_array'>
(286, 286)
float64


Checks: con_mat contains all 286 nodes

In [4]:
print(con_mat.sum())           # total non-zero values
print((con_mat + con_mat.T).sum())  # after symmetry fix

6.0
12.0


Extract edges

In [5]:
# Extract edges
sources, targets = np.where(con_mat > 0)
edges = [(sources[i], targets[i]) for i in range(len(sources)) if sources[i] < targets[i]]

print(f"Number of significant connections: {len(edges)}")

Number of significant connections: 6


plot

In [7]:
from nilearn import plotting

# Convert adjacency matrix to nilearn-friendly format

adjacency_matrix = (con_mat + con_mat.T)  # ensure symmetry

# Get only nodes that participate in significant connections
significant_nodes = np.unique(np.concatenate([sources, targets]))

node_coords_filtered = node_coords[significant_nodes]
adjacency_sub = adjacency_matrix[np.ix_(significant_nodes, significant_nodes)]

plotting.plot_connectome(
    adjacency_sub,
    node_coords_filtered,
    node_color='blue',
    node_size=50,
    edge_vmin=0,
    edge_vmax=1,
    display_mode='lzr',
    colorbar=True,
    edge_kwargs={'color': 'orange'},
    title='NBS Significant Network (Awake > Drowsy, t-threshold=4)',
    output_file= "C:\\Users\\tempu\\Downloads\\research\\labs\\gratton\\Arousal-Project\\visualizations\\nbs_significant_network.tiff"
)
plotting.show()

overlay on brain